# Job Recommendation Engine with Hybrid Ranking, Explainability, and Feedback-Driven Learning using NLP

**Final leakage-safe Kaggle notebook**

This notebook is designed to match the approved research scope while correcting the methodological problems identified in earlier versions.

### Key corrections in this final version

- No artificial or forced accuracy target.
- No `Competency_Aligned_Job_Role` is used as a supervised training target.
- No target is generated from the same skills later given back to the classifier.
- The real dataset field `Applied_Job_Role` remains the only supervised classification label.
- Train, validation, and test sets are separated before role-profile construction.
- Role profiles are built **only from the training split**.
- Hybrid ranking weights are selected **only on the validation split**.
- The held-out test split is touched only once for final evaluation.
- Recommendation metrics are treated as the primary project evaluation.
- Classification accuracy is retained as a diagnostic baseline rather than inflated.
- Explanations are grounded in measurable ranking components.
- Feedback-driven reranking is demonstrated without collecting participant data.

> **Academic prototype only.** This system is not intended for live hiring or automated employment decisions.

### How to read this notebook

Every code section now begins with a short **“What this cell does”** explanation. These comments describe the purpose of the code, why the step is methodologically necessary, and how its output is used later. Decorative separator lines have been removed to keep the notebook clean and professional.


## 1. Research alignment

**Aim:** Design and evaluate an NLP-based job recommendation prototype that combines semantic matching, hybrid ranking, explainable recommendation outputs, and feedback-driven score adjustment using secondary data.

**Primary evaluation:** Precision@K, Recall@K, Mean Reciprocal Rank (MRR), and nDCG@K.

**Secondary diagnostic evaluation:** Accuracy, precision, recall, macro-F1, and top-k classification accuracy for the original `Applied_Job_Role` label.

The dataset contains synthetic resume records and is used only for academic analysis and prototype development.

In [36]:
# 2. Imports and reproducibility

# What this cell does:
# - This cell imports every library used later in the notebook.
# - RANDOM_SEED is fixed so that train/test splits and sampled examples can be reproduced.
# - Plotly is optional: if it is unavailable, the notebook automatically falls back to Matplotlib.

from pathlib import Path
from collections import Counter
import json
import math
import os
import re
import warnings
import zipfile

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    top_k_accuracy_score,
)
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except Exception:
    PLOTLY_AVAILABLE = False

print("Environment ready")
print("Plotly available:", PLOTLY_AVAILABLE)


Environment ready
Plotly available: True


In [37]:
# 3. Professional project configuration

# What this cell does:
# - This cell keeps the important project settings in one place instead of scattering constants throughout the notebook.
# - The configuration defines the random seed, split proportions, evaluation cut-offs, and the expected dataset schema.
# - Output folders are created once so that tables, figures, and deployment files are saved consistently.

class ProjectConfig:
    project_name = "Job Recommendation Engine"
    random_seed = 42
    validation_size = 0.20
    test_size = 0.20
    top_k_values = (1, 3, 5)
    role_profile_top_skills = 10

    dataset_expected_columns = {
        "Name",
        "Experience_Years",
        "Skills",
        "Education",
        "Applied_Job_Role",
    }

CFG = ProjectConfig()

OUTPUT_DIR = Path("/kaggle/working/job_recommendation_final_outputs")
if not Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("./job_recommendation_final_outputs")

FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
DEPLOYMENT_DIR = OUTPUT_DIR / "streamlit_app"

for directory in [OUTPUT_DIR, FIGURE_DIR, TABLE_DIR, DEPLOYMENT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR.resolve())


Output directory: /kaggle/working/job_recommendation_final_outputs


In [38]:
# 4. Dataset discovery

# What this cell does:
# - This function searches common Kaggle/local locations for a CSV that contains the exact columns required by the project.
# - It checks the schema before accepting a file, which prevents the notebook from accidentally loading an unrelated CSV.
# - If no suitable dataset is found, the notebook stops with a clear error rather than continuing with invalid data.

def locate_resume_dataset():
    search_roots = [
        Path("/kaggle/input"),
        Path("."),
    ]

    preferred_names = [
        "large_resume_dataset.csv",
        "ai_resume_matcher.csv",
        "resume_dataset.csv",
    ]

    candidates = []

    for root in search_roots:
        if not root.exists():
            continue

        for name in preferred_names:
            candidates.extend(root.rglob(name))

        candidates.extend(root.rglob("*.csv"))

    checked = set()
    for path in candidates:
        path = path.resolve()
        if path in checked:
            continue
        checked.add(path)

        try:
            sample = pd.read_csv(path, nrows=5)
            if CFG.dataset_expected_columns.issubset(sample.columns):
                return path
        except Exception:
            continue

    raise FileNotFoundError(
        "Could not locate the AI Resume Matcher dataset. "
        "Attach the Kaggle dataset or place large_resume_dataset.csv in the notebook environment."
    )

DATASET_PATH = locate_resume_dataset()
print("Dataset found:", DATASET_PATH)


Dataset found: /kaggle/input/datasets/venkataanjaneyulu0/large-data-csv/large_resume_dataset.csv


In [39]:
# 5. Load, minimise, pseudonymise, and clean data

# What this cell does:
# - The raw dataset is loaded and immediately reduced to only the fields required for the research.
# - Experience is converted to numeric form and impossible values are bounded to a sensible range.
# - Direct names are removed and replaced with project-only Candidate_ID values so the modelling stage does not use personal identifiers.
# - Duplicates and unusable records are removed before any model training takes place.

raw_df = pd.read_csv(DATASET_PATH)

print("Raw shape:", raw_df.shape)
display(raw_df.head())

missing_schema = CFG.dataset_expected_columns - set(raw_df.columns)
assert not missing_schema, f"Missing required columns: {missing_schema}"

working_df = raw_df[
    ["Name", "Experience_Years", "Skills", "Education", "Applied_Job_Role"]
].copy()

# Keep Name only long enough to confirm it exists; it is then removed.
working_df["Experience_Years"] = (
    pd.to_numeric(working_df["Experience_Years"], errors="coerce")
    .fillna(0)
    .clip(lower=0, upper=50)
)

for column in ["Skills", "Education", "Applied_Job_Role"]:
    working_df[column] = (
        working_df[column]
        .fillna("Unknown" if column != "Skills" else "")
        .astype(str)
        .str.strip()
    )

working_df = working_df[
    working_df["Skills"].str.len().gt(0)
    & working_df["Applied_Job_Role"].str.len().gt(0)
].copy()

working_df = working_df.drop_duplicates().reset_index(drop=True)

# Replace any original identifying field with a project-only pseudonymous ID.
working_df["Candidate_ID"] = [
    f"CAND_{idx:05d}" for idx in range(1, len(working_df) + 1)
]

working_df = working_df.drop(columns=["Name"])

print("Clean shape:", working_df.shape)
print("Direct Name column retained:", "Name" in working_df.columns)
display(working_df.head())


Raw shape: (2000, 5)


,Name,Experience_Years,Skills,Education,Applied_Job_Role
0,Candidate_0,1,"Node.js, MongoDB",BCA,Data Scientist
1,Candidate_1,7,"JavaScript, React",B.Com,Product Manager
2,Candidate_2,4,"Angular, TypeScript",BCA,Frontend Developer
3,Candidate_3,0,"Excel, Admin",M.Tech,Backend Developer
4,Candidate_4,15,"Angular, TypeScript",B.Com,DevOps Engineer


Clean shape: (2000, 5)
Direct Name column retained: False


,Experience_Years,Skills,Education,Applied_Job_Role,Candidate_ID
0,1,"Node.js, MongoDB",BCA,Data Scientist,CAND_00001
1,7,"JavaScript, React",B.Com,Product Manager,CAND_00002
2,4,"Angular, TypeScript",BCA,Frontend Developer,CAND_00003
3,0,"Excel, Admin",M.Tech,Backend Developer,CAND_00004
4,15,"Angular, TypeScript",B.Com,DevOps Engineer,CAND_00005


In [40]:
# 6. Feature engineering used throughout the project

# What this cell does:
# - This cell converts the raw Skills text into a reusable Python list and creates simple derived features.
# - Experience_Band makes years of experience easier to interpret, while Skill_Count measures profile breadth.
# - Profile_Text combines only candidate-side information; the target job-role label is deliberately excluded to prevent target leakage.

def parse_skill_list(value):
    if pd.isna(value):
        return []
    return [
        part.strip()
        for part in re.split(r"[,;|/]+", str(value))
        if part.strip()
    ]

def experience_band(years):
    years = float(years)
    if years < 2:
        return "Entry"
    if years < 5:
        return "Early Career"
    if years < 8:
        return "Mid-Level"
    return "Senior"

working_df["Skill_List"] = working_df["Skills"].apply(parse_skill_list)
working_df["Skill_Count"] = working_df["Skill_List"].apply(len)
working_df["Experience_Band"] = working_df["Experience_Years"].apply(experience_band)

# IMPORTANT: this representation uses candidate features only.
# It never inserts the target label into the input text.
working_df["Profile_Text"] = working_df.apply(
    lambda row: (
        f"skills {' '.join(row['Skill_List'])}. "
        f"education {row['Education']}. "
        f"experience {int(row['Experience_Years'])} years. "
        f"experience level {row['Experience_Band']}."
    ),
    axis=1,
)

ALL_SKILLS = sorted({
    skill
    for skill_list in working_df["Skill_List"]
    for skill in skill_list
})

print("Records:", len(working_df))
print("Job roles:", working_df["Applied_Job_Role"].nunique())
print("Education categories:", working_df["Education"].nunique())
print("Unique skills:", len(ALL_SKILLS))


Records: 2000
Job roles: 8
Education categories: 8
Unique skills: 21


## 7. Exploratory Data Analysis

The EDA below examines the dataset before model training. All charts are descriptive only; they are not used to select test-set results.

In [41]:
# 7A. Dataset quality summary

# What this cell does:
# - This table provides a compact quality audit of the cleaned dataset before modelling.
# - The duplicate check uses only hashable scalar/text columns because Skill_List contains Python lists and cannot be hashed by pandas.
# - The summary is exported to CSV so the same values can be cited in the dissertation.

quality_summary = pd.DataFrame({
    "Metric": [
        "Records",
        "Unique job roles",
        "Unique education categories",
        "Unique skills",
        "Mean experience years",
        "Median experience years",
        "Mean skills per resume",
        "Duplicate rows after cleaning",
    ],
    "Value": [
        len(working_df),
        working_df["Applied_Job_Role"].nunique(),
        working_df["Education"].nunique(),
        len(ALL_SKILLS),
        round(working_df["Experience_Years"].mean(), 2),
        round(working_df["Experience_Years"].median(), 2),
        round(working_df["Skill_Count"].mean(), 2),
        int(
            working_df[
                [
                    "Experience_Years",
                    "Skills",
                    "Education",
                    "Applied_Job_Role",
                    "Candidate_ID",
                    "Skill_Count",
                    "Experience_Band",
                    "Profile_Text",
                ]
            ]
            .duplicated()
            .sum()
        ),
    ],
})

display(quality_summary)
quality_summary.to_csv(TABLE_DIR / "dataset_quality_summary.csv", index=False)


,Metric,Value
0,Records,2000.00
1,Unique job roles,8.00
2,Unique education categories,8.00
3,Unique skills,21.00
4,Mean experience years,7.55
5,Median experience years,8.00
6,Mean skills per resume,2.11
7,Duplicate rows after cleaning,0.00


In [42]:
# 7B. Job role distribution

# What this cell does:
# - This chart shows how many records belong to each original Applied_Job_Role class.
# - It helps identify class imbalance before classification and recommendation evaluation.
# - The underlying counts are also saved as a CSV table.

role_counts = (
    working_df["Applied_Job_Role"]
    .value_counts()
    .rename_axis("Job Role")
    .reset_index(name="Count")
)

if PLOTLY_AVAILABLE:
    fig = px.bar(
        role_counts.sort_values("Count"),
        x="Count",
        y="Job Role",
        orientation="h",
        text="Count",
        title="Distribution of applied job roles",
    )
    fig.update_layout(template="plotly_white", height=500)
    fig.show()
else:
    plt.figure(figsize=(11, 6))
    plt.barh(role_counts["Job Role"], role_counts["Count"])
    plt.title("Distribution of applied job roles")
    plt.show()

role_counts.to_csv(TABLE_DIR / "job_role_distribution.csv", index=False)


In [43]:
# 7C. Education distribution

# What this cell does:
# - This visualization shows how education categories are distributed across the synthetic resume dataset.
# - It is descriptive only and does not influence model selection or test-set evaluation.

education_counts = (
    working_df["Education"]
    .value_counts()
    .rename_axis("Education")
    .reset_index(name="Count")
)

if PLOTLY_AVAILABLE:
    fig = px.bar(
        education_counts,
        x="Education",
        y="Count",
        text="Count",
        title="Education profile distribution",
    )
    fig.update_layout(template="plotly_white", xaxis_tickangle=-25)
    fig.show()
else:
    plt.figure(figsize=(10, 5))
    plt.bar(education_counts["Education"], education_counts["Count"])
    plt.xticks(rotation=25)
    plt.title("Education profile distribution")
    plt.show()


In [44]:
# 7D. Experience distribution and experience bands

# What this cell does:
# - The histogram shows the continuous distribution of years of experience.
# - The experience-band chart provides an easier high-level view of entry, early-career, mid-level, and senior profiles.
# - These charts help explain the population used by the recommender.

if PLOTLY_AVAILABLE:
    fig = px.histogram(
        working_df,
        x="Experience_Years",
        nbins=16,
        title="Distribution of candidate experience",
        marginal="box",
    )
    fig.update_layout(template="plotly_white")
    fig.show()

    band_counts = (
        working_df["Experience_Band"]
        .value_counts()
        .rename_axis("Experience Band")
        .reset_index(name="Count")
    )
    fig = px.pie(
        band_counts,
        names="Experience Band",
        values="Count",
        hole=0.45,
        title="Experience-band composition",
    )
    fig.update_layout(template="plotly_white")
    fig.show()
else:
    plt.figure(figsize=(10, 5))
    plt.hist(working_df["Experience_Years"], bins=16)
    plt.title("Distribution of candidate experience")
    plt.show()


In [45]:
# 7E. Top skills

# What this cell does:
# - All parsed skill lists are flattened and counted to identify the most frequent skills in the dataset.
# - The top-25 chart helps explain which competencies dominate the available synthetic resumes.
# - These frequencies are descriptive and are not calculated from the test labels.

skill_counter = Counter(
    skill
    for skill_list in working_df["Skill_List"]
    for skill in skill_list
)

top_skills_df = pd.DataFrame(
    skill_counter.most_common(25),
    columns=["Skill", "Count"],
)

if PLOTLY_AVAILABLE:
    fig = px.bar(
        top_skills_df.sort_values("Count"),
        x="Count",
        y="Skill",
        orientation="h",
        text="Count",
        title="Top 25 skills in the dataset",
    )
    fig.update_layout(template="plotly_white", height=700)
    fig.show()
else:
    plt.figure(figsize=(11, 8))
    plt.barh(top_skills_df["Skill"], top_skills_df["Count"])
    plt.title("Top 25 skills in the dataset")
    plt.show()

top_skills_df.to_csv(TABLE_DIR / "top_skills.csv", index=False)


In [46]:
# 7F. Mean experience and skill count by applied role

# What this cell does:
# - This aggregation compares average experience and average number of listed skills across the original job-role labels.
# - It provides evidence about whether the role labels are meaningfully separated by candidate characteristics.

role_profile_summary = (
    working_df.groupby("Applied_Job_Role")
    .agg(
        Mean_Experience=("Experience_Years", "mean"),
        Median_Experience=("Experience_Years", "median"),
        Mean_Skill_Count=("Skill_Count", "mean"),
        Records=("Candidate_ID", "count"),
    )
    .reset_index()
)

display(role_profile_summary.round(2))

if PLOTLY_AVAILABLE:
    fig = px.bar(
        role_profile_summary,
        x="Applied_Job_Role",
        y=["Mean_Experience", "Mean_Skill_Count"],
        barmode="group",
        title="Profile characteristics by applied role",
    )
    fig.update_layout(template="plotly_white", xaxis_tickangle=-25)
    fig.show()


,Applied_Job_Role,Mean_Experience,Median_Experience,Mean_Skill_Count,Records
0,Backend Developer,7.68,7.0,2.10,271
1,Data Analyst,7.20,7.0,2.12,252
2,Data Scientist,7.26,7.0,2.10,225
3,DevOps Engineer,7.34,7.0,2.12,226
4,Embedded Engineer,7.11,7.0,2.12,264
5,Frontend Developer,8.13,8.0,2.09,263
6,Product Manager,7.74,8.0,2.12,246
7,Virtual Assistant,7.91,8.0,2.11,253


In [47]:
# 7G. Role × skill prevalence heatmap

# What this cell does:
# - For each role, this cell calculates the proportion of candidates containing each of the 15 most common skills.
# - The heatmap helps visually assess whether certain skills are concentrated in particular role labels.
# - Weak separation here can help explain modest classification accuracy later.

top_15_skills = top_skills_df.head(15)["Skill"].tolist()

heat_rows = []
for role, group in working_df.groupby("Applied_Job_Role"):
    role_size = len(group)
    for skill in top_15_skills:
        prevalence = group["Skill_List"].apply(lambda values: skill in values).mean()
        heat_rows.append({
            "Role": role,
            "Skill": skill,
            "Prevalence": prevalence,
        })

heat_df = pd.DataFrame(heat_rows)
heat_pivot = heat_df.pivot(index="Role", columns="Skill", values="Prevalence")

if PLOTLY_AVAILABLE:
    fig = px.imshow(
        heat_pivot,
        aspect="auto",
        title="Skill prevalence by applied role",
        labels={"color": "Prevalence"},
    )
    fig.update_layout(template="plotly_white", height=650)
    fig.show()
else:
    plt.figure(figsize=(14, 7))
    plt.imshow(heat_pivot.values, aspect="auto")
    plt.xticks(range(len(heat_pivot.columns)), heat_pivot.columns, rotation=60, ha="right")
    plt.yticks(range(len(heat_pivot.index)), heat_pivot.index)
    plt.title("Skill prevalence by applied role")
    plt.colorbar()
    plt.show()


## 8. Leakage-safe train / validation / test split

The split happens **before**:

- vectorizer fitting,
- classifier fitting,
- role-profile construction,
- semantic profile construction,
- hybrid-weight selection.

This prevents information from validation or test candidates leaking into the training representation.

In [48]:
# 8A. Stratified 60/20/20 split

# What this cell does:
# - The cleaned dataset is divided into training, validation, and test sets while preserving the class distribution.
# - Training data is used to learn model parameters and role profiles.
# - Validation data is used only to choose hybrid-ranking weights.
# - Test data is kept untouched until final evaluation, preventing optimistic performance estimates.

train_df, holdout_df = train_test_split(
    working_df,
    test_size=CFG.validation_size + CFG.test_size,
    stratify=working_df["Applied_Job_Role"],
    random_state=CFG.random_seed,
)

relative_test_fraction = CFG.test_size / (CFG.validation_size + CFG.test_size)

validation_df, test_df = train_test_split(
    holdout_df,
    test_size=relative_test_fraction,
    stratify=holdout_df["Applied_Job_Role"],
    random_state=CFG.random_seed,
)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

split_summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Records": [len(train_df), len(validation_df), len(test_df)],
})

display(split_summary)
split_summary.to_csv(TABLE_DIR / "split_summary.csv", index=False)


Train: (1200, 9)
Validation: (400, 9)
Test: (400, 9)


,Split,Records
0,Train,1200
1,Validation,400
2,Test,400


## 9. Baseline supervised classification on the real label

The real `Applied_Job_Role` field is kept as the classification target.  
No synthetic competency label is created.

A low classification score is reported honestly because it reflects the information content of this synthetic dataset.

In [49]:
# 9A. TF-IDF fitted on training split only

# What this cell does:
# - TF-IDF converts profile text into numerical features that capture informative words and two-word phrases.
# - The vectorizer is fitted only on the training split so vocabulary from validation/test records cannot leak into training.
# - The same fitted vectorizer is then used to transform validation and test data.

classification_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    sublinear_tf=True,
    max_features=8000,
)

X_train_text = classification_vectorizer.fit_transform(train_df["Profile_Text"])
X_validation_text = classification_vectorizer.transform(validation_df["Profile_Text"])
X_test_text = classification_vectorizer.transform(test_df["Profile_Text"])

y_train = train_df["Applied_Job_Role"]
y_validation = validation_df["Applied_Job_Role"]
y_test = test_df["Applied_Job_Role"]

print("Training vocabulary size:", len(classification_vectorizer.vocabulary_))


Training vocabulary size: 109


In [50]:
# 9B. Compare honest classification baselines

# What this cell does:
# - A DummyClassifier provides a simple majority-class reference point.
# - Logistic Regression is the main supervised baseline for predicting the original Applied_Job_Role.
# - Models are compared on validation accuracy and macro-averaged precision, recall, and F1 so minority classes are not ignored.
# - The best validation model is selected before the test split is evaluated.

classification_models = {
    "Dummy Majority": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(
        max_iter=4000,
        class_weight="balanced",
        C=1.0,
        random_state=CFG.random_seed,
    ),
}

classification_rows = []
trained_classifiers = {}

for model_name, estimator in classification_models.items():
    estimator.fit(X_train_text, y_train)
    val_pred = estimator.predict(X_validation_text)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_validation,
        val_pred,
        average="macro",
        zero_division=0,
    )

    classification_rows.append({
        "Model": model_name,
        "Validation Accuracy": accuracy_score(y_validation, val_pred),
        "Validation Macro Precision": precision,
        "Validation Macro Recall": recall,
        "Validation Macro F1": f1,
    })

    trained_classifiers[model_name] = estimator

classification_validation_df = (
    pd.DataFrame(classification_rows)
    .sort_values(["Validation Macro F1", "Validation Accuracy"], ascending=False)
    .reset_index(drop=True)
)

display(classification_validation_df.round(4))
classification_validation_df.to_csv(
    TABLE_DIR / "classification_validation_metrics.csv",
    index=False,
)

BEST_CLASSIFIER_NAME = classification_validation_df.iloc[0]["Model"]
best_classifier = trained_classifiers[BEST_CLASSIFIER_NAME]

print("Selected classification baseline:", BEST_CLASSIFIER_NAME)


,Model,Validation Accuracy,Validation Macro Precision,Validation Macro Recall,Validation Macro F1
0,Logistic Regression,0.140,0.1448,0.1392,0.1400
1,Dummy Majority,0.135,0.0169,0.1250,0.0297


Selected classification baseline: Logistic Regression


In [51]:
# 9C. Final classification test metrics

# What this cell does:
# - The selected classifier is evaluated once on the untouched test set.
# - Accuracy measures exact class prediction, while macro metrics give equal importance to every job-role class.
# - Top-3 and Top-5 accuracy are also reported because recommendation systems often care whether the relevant role appears near the top rather than only at rank 1.

test_pred = best_classifier.predict(X_test_text)

test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(
    y_test,
    test_pred,
    average="macro",
    zero_division=0,
)

classification_test_metrics = pd.DataFrame([{
    "Model": BEST_CLASSIFIER_NAME,
    "Test Accuracy": accuracy_score(y_test, test_pred),
    "Test Macro Precision": test_precision,
    "Test Macro Recall": test_recall,
    "Test Macro F1": test_f1,
}])

if hasattr(best_classifier, "predict_proba"):
    test_prob = best_classifier.predict_proba(X_test_text)
    classes = list(best_classifier.classes_)
    encoded_test = pd.Categorical(y_test, categories=classes).codes

    for k in [3, 5]:
        classification_test_metrics[f"Top-{k} Accuracy"] = top_k_accuracy_score(
            encoded_test,
            test_prob,
            k=k,
            labels=np.arange(len(classes)),
        )

display(classification_test_metrics.round(4))
classification_test_metrics.to_csv(
    TABLE_DIR / "classification_test_metrics.csv",
    index=False,
)

report_df = pd.DataFrame(
    classification_report(
        y_test,
        test_pred,
        output_dict=True,
        zero_division=0,
    )
).T

display(report_df.round(4))
report_df.to_csv(TABLE_DIR / "classification_report.csv")


,Model,Test Accuracy,Test Macro Precision,Test Macro Recall,Test Macro F1,Top-3 Accuracy,Top-5 Accuracy
0,Logistic Regression,0.155,0.155,0.1532,0.1533,0.395,0.6175


,precision,recall,f1-score,support
Backend Developer,0.2000,0.1852,0.1923,54.000
Data Analyst,0.2449,0.2353,0.2400,51.000
Data Scientist,0.0833,0.0889,0.0860,45.000
DevOps Engineer,0.1429,0.1111,0.1250,45.000
Embedded Engineer,0.1111,0.1132,0.1121,53.000
Frontend Developer,0.1818,0.1923,0.1869,52.000
Product Manager,0.1395,0.1200,0.1290,50.000
Virtual Assistant,0.1364,0.1800,0.1552,50.000
accuracy,0.1550,0.1550,0.1550,0.155
macro avg,0.1550,0.1532,0.1533,400.000


In [52]:
# 9D. Confusion matrix

# What this cell does:
# - The confusion matrix shows which job roles are most often confused with one another.
# - This is especially useful when overall accuracy is modest because it reveals the pattern of model errors instead of hiding them behind a single number.

class_labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, test_pred, labels=class_labels)

if PLOTLY_AVAILABLE:
    fig = px.imshow(
        cm,
        x=class_labels,
        y=class_labels,
        text_auto=True,
        aspect="auto",
        title=f"Confusion matrix — {BEST_CLASSIFIER_NAME}",
        labels={"x": "Predicted role", "y": "Actual role", "color": "Count"},
    )
    fig.update_layout(template="plotly_white", height=650)
    fig.show()
else:
    plt.figure(figsize=(10, 8))
    plt.imshow(cm)
    plt.xticks(range(len(class_labels)), class_labels, rotation=45, ha="right")
    plt.yticks(range(len(class_labels)), class_labels)
    plt.title(f"Confusion matrix — {BEST_CLASSIFIER_NAME}")
    plt.colorbar()
    plt.show()


## 10. Training-only role profiles

The recommender uses role profiles derived **only from training candidates**.  
No validation or test candidate contributes to the role profiles.

In [53]:
# 10A. Build role profiles from training data only

# What this cell does:
# - Each job-role profile is constructed only from training records to avoid using validation/test information.
# - The profile stores the most common skills, common education categories, median experience, and the number of training examples for that role.
# - These role profiles become the reference points used by the hybrid recommender.

def most_common_skills(series, n=10):
    counter = Counter(
        skill
        for skill_list in series
        for skill in skill_list
    )
    return [skill for skill, _ in counter.most_common(n)]

role_profiles = []

for role, group in train_df.groupby("Applied_Job_Role"):
    top_skills = most_common_skills(
        group["Skill_List"],
        CFG.role_profile_top_skills,
    )

    common_education = (
        group["Education"]
        .value_counts()
        .head(3)
        .index
        .tolist()
    )

    median_experience = float(group["Experience_Years"].median())

    role_profile_text = (
        f"role {role}. "
        f"important skills {' '.join(top_skills)}. "
        f"common education {' '.join(common_education)}. "
        f"typical experience {median_experience:.1f} years."
    )

    role_profiles.append({
        "Role": role,
        "Top_Skills": top_skills,
        "Common_Education": common_education,
        "Median_Experience": median_experience,
        "Training_Records": len(group),
        "Role_Profile_Text": role_profile_text,
    })

role_profiles_df = (
    pd.DataFrame(role_profiles)
    .sort_values("Role")
    .reset_index(drop=True)
)

display(role_profiles_df)
role_profiles_df.to_csv(TABLE_DIR / "training_only_role_profiles.csv", index=False)


,Role,Top_Skills,Common_Education,Median_Experience,Training_Records,Role_Profile_Text
0,Backend Developer,"[Node.js, MongoDB, AWS, DevOps, R, Statistics,...","[M.Tech, B.Com, B.Sc in IT]",8.0,163,role Backend Developer. important skills Node....
1,Data Analyst,"[Angular, TypeScript, Python, SQL, ML, R, Stat...","[BCA, MCA, B.Com]",7.0,151,role Data Analyst. important skills Angular Ty...
2,Data Scientist,"[Excel, Admin, PHP, Laravel, Angular, TypeScri...","[BCA, B.Tech in CS, Diploma in CS]",7.0,135,role Data Scientist. important skills Excel Ad...
3,DevOps Engineer,"[Python, SQL, ML, JavaScript, React, Excel, Ad...","[BCA, B.Sc in IT, Diploma in CS]",7.0,136,role DevOps Engineer. important skills Python ...
4,Embedded Engineer,"[C++, IoT, PHP, Laravel, Excel, Admin, Python,...","[B.Tech in CS, B.Sc in IT, B.Com]",7.0,158,role Embedded Engineer. important skills C++ I...
5,Frontend Developer,"[Node.js, MongoDB, Angular, TypeScript, C++, I...","[M.Tech, MCA, Diploma in CS]",8.0,158,role Frontend Developer. important skills Node...
6,Product Manager,"[Python, SQL, ML, JavaScript, React, AWS, DevO...","[Diploma in CS, MCA, B.Sc in IT]",8.0,147,role Product Manager. important skills Python ...
7,Virtual Assistant,"[JavaScript, React, Python, SQL, ML, AWS, DevO...","[MBA, B.Sc in IT, BCA]",8.0,152,role Virtual Assistant. important skills JavaS...


In [54]:
# 10B. Independent semantic representation

# What this cell does:
# - A separate TF-IDF model is fitted to training-derived candidate text and training-derived role-profile text.
# - This semantic model measures how similar a new candidate profile is to each role profile.
# - No test labels are used to build the semantic representation.

# Fitted only to TRAIN-derived role profile text plus TRAIN candidate text.
semantic_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
    max_features=5000,
)

semantic_training_corpus = (
    train_df["Profile_Text"].tolist()
    + role_profiles_df["Role_Profile_Text"].tolist()
)

semantic_vectorizer.fit(semantic_training_corpus)
role_profile_matrix = semantic_vectorizer.transform(
    role_profiles_df["Role_Profile_Text"]
)

print("Semantic vocabulary size:", len(semantic_vectorizer.vocabulary_))


Semantic vocabulary size: 208


## 11. Hybrid recommender

The final score combines:

- **Skill overlap** — direct overlap with training-derived role skills.
- **Semantic similarity** — TF-IDF similarity to the training-derived role profile.
- **Model probability** — probability from the honest original-label classifier.
- **Experience fit** — closeness to the training-derived median experience.
- **Education fit** — whether education is common in the training-derived role profile.

No test label is used when generating recommendations.

In [55]:
# 11A. Recommendation functions

# What this cell does:
# - This is the core hybrid recommendation engine.
# - For each candidate-role pair it calculates skill overlap, semantic similarity, classifier probability, experience fit, education fit, and optional feedback.
# - The weighted components are combined into one Hybrid_Score and sorted from highest to lowest.
# - Matched and missing skills are retained so each recommendation can be explained transparently.

ROLE_NAMES = role_profiles_df["Role"].tolist()
CLASS_TO_INDEX = {
    cls: idx
    for idx, cls in enumerate(best_classifier.classes_)
} if hasattr(best_classifier, "classes_") else {}

def safe_model_probability_map(profile_text):
    vector = classification_vectorizer.transform([profile_text])

    if hasattr(best_classifier, "predict_proba"):
        probabilities = best_classifier.predict_proba(vector)[0]
        return {
            str(role): float(prob)
            for role, prob in zip(best_classifier.classes_, probabilities)
        }

    predicted = str(best_classifier.predict(vector)[0])
    return {role: float(role == predicted) for role in ROLE_NAMES}

def experience_fit(candidate_years, reference_years):
    difference = abs(float(candidate_years) - float(reference_years))
    return max(0.0, 1.0 - difference / 15.0)

def education_fit(candidate_education, common_education):
    return 1.0 if candidate_education in common_education else 0.35

def recommend_candidate(row, weights, feedback=None, top_n=None):
    feedback = feedback or {}
    candidate_skills = set(row["Skill_List"])

    probability_map = safe_model_probability_map(row["Profile_Text"])

    candidate_semantic_vector = semantic_vectorizer.transform(
        [row["Profile_Text"]]
    )
    semantic_scores = cosine_similarity(
        candidate_semantic_vector,
        role_profile_matrix,
    ).flatten()

    recommendation_rows = []

    for idx, profile in role_profiles_df.iterrows():
        role = profile["Role"]
        role_skills = set(profile["Top_Skills"])

        matched_skills = sorted(candidate_skills & role_skills)
        missing_skills = sorted(role_skills - candidate_skills)

        skill_score = len(matched_skills) / max(len(role_skills), 1)
        semantic_score = float(semantic_scores[idx])
        probability_score = float(probability_map.get(role, 0.0))
        experience_score = experience_fit(
            row["Experience_Years"],
            profile["Median_Experience"],
        )
        education_score = education_fit(
            row["Education"],
            profile["Common_Education"],
        )
        feedback_score = float(feedback.get(role, 0.0))

        base_score = (
            weights["skill"] * skill_score
            + weights["semantic"] * semantic_score
            + weights["model_probability"] * probability_score
            + weights["experience"] * experience_score
            + weights["education"] * education_score
        )

        recommendation_rows.append({
            "Recommended_Role": role,
            "Base_Score": base_score,
            "Hybrid_Score": base_score + feedback_score,
            "Skill_Fit": skill_score,
            "Semantic_Similarity": semantic_score,
            "Model_Probability": probability_score,
            "Experience_Fit": experience_score,
            "Education_Fit": education_score,
            "Feedback_Adjustment": feedback_score,
            "Matched_Skills": matched_skills,
            "Missing_Skills": missing_skills,
        })

    result = (
        pd.DataFrame(recommendation_rows)
        .sort_values("Hybrid_Score", ascending=False)
        .reset_index(drop=True)
    )
    result.insert(0, "Rank", np.arange(1, len(result) + 1))

    if top_n is not None:
        return result.head(top_n)

    return result


In [56]:
# 11B. Ranking metric functions

# What this cell does:
# - These functions evaluate recommendation quality based on the rank of the candidate's original Applied_Job_Role.
# - MRR rewards systems that place the relevant role very near the top.
# - Recall@K checks whether the relevant role appears within the top K recommendations.
# - Precision@K and nDCG@K provide additional ranking-quality perspectives.

def reciprocal_rank(rank):
    return 1.0 / rank

def ndcg_at_k(rank, k):
    if rank > k:
        return 0.0
    return 1.0 / math.log2(rank + 1)

def evaluate_recommender(dataframe, weights, k_values=(1, 3, 5)):
    candidate_rows = []

    for _, row in dataframe.iterrows():
        recommendations = recommend_candidate(
            row,
            weights=weights,
            top_n=None,
        )

        actual_role = row["Applied_Job_Role"]
        matching_ranks = recommendations.index[
            recommendations["Recommended_Role"].eq(actual_role)
        ].tolist()

        if not matching_ranks:
            rank = len(recommendations) + 1
        else:
            rank = matching_ranks[0] + 1

        item = {
            "Candidate_ID": row["Candidate_ID"],
            "Actual_Role": actual_role,
            "Rank": rank,
            "MRR_Component": reciprocal_rank(rank),
        }

        for k in k_values:
            hit = float(rank <= k)
            item[f"Recall@{k}"] = hit
            item[f"Precision@{k}"] = hit / k
            item[f"nDCG@{k}"] = ndcg_at_k(rank, k)

        candidate_rows.append(item)

    detail_df = pd.DataFrame(candidate_rows)

    summary = {
        "MRR": detail_df["MRR_Component"].mean(),
    }

    for k in k_values:
        summary[f"Precision@{k}"] = detail_df[f"Precision@{k}"].mean()
        summary[f"Recall@{k}"] = detail_df[f"Recall@{k}"].mean()
        summary[f"nDCG@{k}"] = detail_df[f"nDCG@{k}"].mean()

    return pd.DataFrame([summary]), detail_df


## 12. Validation-only weight selection

The hybrid weights are selected using the **validation split only**.  
The final test set remains untouched during this step.

In [57]:
# 12A. Candidate weight configurations

# What this cell does:
# - Several reasonable hybrid-weight combinations are tested on the validation split only.
# - Each configuration changes the relative importance of skills, semantic similarity, classifier probability, experience, and education.
# - The best configuration is selected using validation nDCG@5 and MRR, not test performance.

candidate_weight_sets = [
    {
        "Name": "Balanced",
        "skill": 0.35,
        "semantic": 0.35,
        "model_probability": 0.10,
        "experience": 0.15,
        "education": 0.05,
    },
    {
        "Name": "Semantic Focus",
        "skill": 0.30,
        "semantic": 0.40,
        "model_probability": 0.10,
        "experience": 0.15,
        "education": 0.05,
    },
    {
        "Name": "Skill Focus",
        "skill": 0.40,
        "semantic": 0.30,
        "model_probability": 0.10,
        "experience": 0.15,
        "education": 0.05,
    },
    {
        "Name": "Low Model Influence",
        "skill": 0.35,
        "semantic": 0.40,
        "model_probability": 0.05,
        "experience": 0.15,
        "education": 0.05,
    },
    {
        "Name": "Higher Model Influence",
        "skill": 0.35,
        "semantic": 0.30,
        "model_probability": 0.15,
        "experience": 0.15,
        "education": 0.05,
    },
]

validation_weight_rows = []

for candidate_weights in candidate_weight_sets:
    weight_name = candidate_weights["Name"]
    actual_weights = {
        key: value
        for key, value in candidate_weights.items()
        if key != "Name"
    }

    validation_summary, _ = evaluate_recommender(
        validation_df,
        actual_weights,
        CFG.top_k_values,
    )

    validation_weight_rows.append({
        "Weight Configuration": weight_name,
        **actual_weights,
        **validation_summary.iloc[0].to_dict(),
    })

validation_weight_results = (
    pd.DataFrame(validation_weight_rows)
    .sort_values(["nDCG@5", "MRR"], ascending=False)
    .reset_index(drop=True)
)

display(validation_weight_results.round(4))
validation_weight_results.to_csv(
    TABLE_DIR / "validation_weight_selection.csv",
    index=False,
)

BEST_WEIGHT_NAME = validation_weight_results.iloc[0]["Weight Configuration"]

BEST_WEIGHTS = {
    key: float(validation_weight_results.iloc[0][key])
    for key in [
        "skill",
        "semantic",
        "model_probability",
        "experience",
        "education",
    ]
}

print("Selected configuration:", BEST_WEIGHT_NAME)
print("Selected weights:", BEST_WEIGHTS)


,Weight Configuration,skill,semantic,model_probability,experience,education,MRR,Precision@1,Recall@1,nDCG@1,Precision@3,Recall@3,nDCG@3,Precision@5,Recall@5,nDCG@5
0,Skill Focus,0.40,0.30,0.10,0.15,0.05,0.3112,0.0950,0.0950,0.0950,0.1075,0.3225,0.2232,0.1240,0.6200,0.3451
1,Balanced,0.35,0.35,0.10,0.15,0.05,0.3122,0.0975,0.0975,0.0975,0.1092,0.3275,0.2259,0.1235,0.6175,0.3448
2,Low Model Influence,0.35,0.40,0.05,0.15,0.05,0.3124,0.0950,0.0950,0.0950,0.1083,0.3250,0.2254,0.1225,0.6125,0.3433
3,Higher Model Influence,0.35,0.30,0.15,0.15,0.05,0.3147,0.1000,0.1000,0.1000,0.1100,0.3300,0.2294,0.1210,0.6050,0.3425
4,Semantic Focus,0.30,0.40,0.10,0.15,0.05,0.3134,0.1000,0.1000,0.1000,0.1083,0.3250,0.2266,0.1200,0.6000,0.3394


Selected configuration: Skill Focus
Selected weights: {'skill': 0.4, 'semantic': 0.3, 'model_probability': 0.1, 'experience': 0.15, 'education': 0.05}


In [58]:
# 12B. Premium validation comparison chart

# What this cell does:
# - This chart compares the main validation ranking metrics across candidate weight configurations.
# - It makes the weight-selection decision visible and auditable rather than presenting a single unexplained configuration.

if PLOTLY_AVAILABLE:
    chart_df = validation_weight_results[
        ["Weight Configuration", "MRR", "nDCG@5", "Recall@5"]
    ].melt(
        id_vars="Weight Configuration",
        var_name="Metric",
        value_name="Score",
    )

    fig = px.bar(
        chart_df,
        x="Weight Configuration",
        y="Score",
        color="Metric",
        barmode="group",
        title="Validation-only hybrid weight selection",
    )
    fig.update_layout(
        template="plotly_white",
        xaxis_tickangle=-20,
        yaxis_range=[0, 1],
    )
    fig.show()


## 13. Final held-out test evaluation

This is the first point at which the test split is used for hybrid-recommender evaluation.

In [59]:
# 13A. Final hybrid recommender metrics

# What this cell does:
# - After all design choices are fixed, the selected hybrid recommender is evaluated on the untouched test split.
# - These values are the primary final recommendation results of the project.
# - No metric is altered or forced to reach a predetermined target.

hybrid_test_summary, hybrid_test_detail = evaluate_recommender(
    test_df,
    BEST_WEIGHTS,
    CFG.top_k_values,
)

display(hybrid_test_summary.round(4))

hybrid_test_summary.to_csv(
    TABLE_DIR / "hybrid_test_metrics.csv",
    index=False,
)
hybrid_test_detail.to_csv(
    TABLE_DIR / "hybrid_test_candidate_ranks.csv",
    index=False,
)

print(
    "Interpretation: the metrics above are measured on the untouched test split. "
    "No test result has been forced to a preferred value."
)


,MRR,Precision@1,Recall@1,nDCG@1,Precision@3,Recall@3,nDCG@3,Precision@5,Recall@5,nDCG@5
0,0.3595,0.145,0.145,0.145,0.1333,0.4,0.2915,0.1275,0.6375,0.3885


Interpretation: the metrics above are measured on the untouched test split. No test result has been forced to a preferred value.


In [60]:
# 13B. Compare hybrid ranking with simpler baselines

# What this cell does:
# - The final hybrid recommender is compared with skill-only, semantic-only, and classifier-only ranking approaches.
# - This comparison shows whether combining multiple signals provides value beyond any single component.

baseline_weight_sets = {
    "Skill-only": {
        "skill": 1.0,
        "semantic": 0.0,
        "model_probability": 0.0,
        "experience": 0.0,
        "education": 0.0,
    },
    "Semantic-only": {
        "skill": 0.0,
        "semantic": 1.0,
        "model_probability": 0.0,
        "experience": 0.0,
        "education": 0.0,
    },
    "Classifier-only": {
        "skill": 0.0,
        "semantic": 0.0,
        "model_probability": 1.0,
        "experience": 0.0,
        "education": 0.0,
    },
    f"Hybrid — {BEST_WEIGHT_NAME}": BEST_WEIGHTS,
}

comparison_rows = []

for method_name, method_weights in baseline_weight_sets.items():
    summary, _ = evaluate_recommender(
        test_df,
        method_weights,
        CFG.top_k_values,
    )
    comparison_rows.append({
        "Method": method_name,
        **summary.iloc[0].to_dict(),
    })

ranking_comparison_df = (
    pd.DataFrame(comparison_rows)
    .sort_values(["nDCG@5", "MRR"], ascending=False)
    .reset_index(drop=True)
)

display(ranking_comparison_df.round(4))
ranking_comparison_df.to_csv(
    TABLE_DIR / "ranking_method_comparison.csv",
    index=False,
)

if PLOTLY_AVAILABLE:
    chart_df = ranking_comparison_df[
        ["Method", "MRR", "nDCG@5", "Recall@5"]
    ].melt(
        id_vars="Method",
        var_name="Metric",
        value_name="Score",
    )

    fig = px.bar(
        chart_df,
        x="Method",
        y="Score",
        color="Metric",
        barmode="group",
        title="Final held-out ranking comparison",
    )
    fig.update_layout(
        template="plotly_white",
        xaxis_tickangle=-20,
        yaxis_range=[0, 1],
    )
    fig.show()


,Method,MRR,Precision@1,Recall@1,nDCG@1,Precision@3,Recall@3,nDCG@3,Precision@5,Recall@5,nDCG@5
0,Hybrid — Skill Focus,0.3595,0.145,0.145,0.145,0.1333,0.4000,0.2915,0.1275,0.6375,0.3885
1,Classifier-only,0.3617,0.155,0.155,0.155,0.1317,0.3950,0.2914,0.1235,0.6175,0.3823
2,Semantic-only,0.3477,0.130,0.130,0.130,0.1308,0.3925,0.2796,0.1270,0.6350,0.3781
3,Skill-only,0.3489,0.140,0.140,0.140,0.1233,0.3700,0.2710,0.1240,0.6200,0.3736


In [61]:
# 13C. Rank distribution

# What this cell does:
# - This table and chart show the exact rank at which the original applied role appears for test candidates.
# - It provides an intuitive view of how often the desired role is ranked first, near the top, or lower in the list.

rank_distribution = (
    hybrid_test_detail["Rank"]
    .value_counts()
    .sort_index()
    .rename_axis("Rank")
    .reset_index(name="Candidates")
)

display(rank_distribution)

if PLOTLY_AVAILABLE:
    fig = px.bar(
        rank_distribution,
        x="Rank",
        y="Candidates",
        text="Candidates",
        title="Rank of the originally applied role in final recommendations",
    )
    fig.update_layout(template="plotly_white")
    fig.show()


,Rank,Candidates
0,1,58
1,2,58
2,3,44
3,4,47
4,5,48
5,6,40
6,7,55
7,8,50


## 14. Explainability demonstration

The explanation layer does not generate unsupported natural-language reasoning.  
It reports the actual score components used by the recommender.

In [62]:
# 14A. Explain one held-out candidate

# What this cell does:
# - A single candidate is sampled from the held-out test set only for explanation and demonstration.
# - The top five recommendations and all score components are displayed so a reader can see exactly why each role received its ranking.
# - Sampling uses the fixed random seed, making the example reproducible.

sample_candidate = test_df.sample(1, random_state=CFG.random_seed).iloc[0]

sample_recommendations = recommend_candidate(
    sample_candidate,
    weights=BEST_WEIGHTS,
    top_n=5,
)

display_columns = [
    "Rank",
    "Recommended_Role",
    "Hybrid_Score",
    "Skill_Fit",
    "Semantic_Similarity",
    "Model_Probability",
    "Experience_Fit",
    "Education_Fit",
    "Matched_Skills",
    "Missing_Skills",
]

print("Candidate:", sample_candidate["Candidate_ID"])
print("Original applied role:", sample_candidate["Applied_Job_Role"])
print("Skills:", sample_candidate["Skill_List"])
print("Education:", sample_candidate["Education"])
print("Experience:", sample_candidate["Experience_Years"])

display(sample_recommendations[display_columns])

if PLOTLY_AVAILABLE:
    fig = px.bar(
        sample_recommendations.sort_values("Hybrid_Score"),
        x="Hybrid_Score",
        y="Recommended_Role",
        orientation="h",
        text="Hybrid_Score",
        title="Top-5 explainable recommendations for one held-out candidate",
    )
    fig.update_layout(template="plotly_white")
    fig.show()


Candidate: CAND_00509
Original applied role: DevOps Engineer
Skills: ['Java', 'Spring']
Education: B.Com
Experience: 5


,Rank,Recommended_Role,Hybrid_Score,Skill_Fit,Semantic_Similarity,Model_Probability,Experience_Fit,Education_Fit,Matched_Skills,Missing_Skills
0,1,Frontend Developer,0.262976,0.2,0.109286,0.126906,0.800000,0.35,"[Java, Spring]","[Admin, Angular, C++, Excel, IoT, MongoDB, Nod..."
1,2,Embedded Engineer,0.258509,0.1,0.073557,0.164419,0.866667,1.00,[Java],"[Admin, C++, Excel, IoT, Laravel, ML, PHP, Pyt..."
2,3,Data Analyst,0.202102,0.0,0.044913,0.086276,0.866667,1.00,[],"[Admin, Angular, Excel, ML, PHP, Python, R, SQ..."
3,4,Backend Developer,0.199920,0.0,0.044149,0.166754,0.800000,1.00,[],"[AWS, Admin, C++, DevOps, Excel, IoT, MongoDB,..."
4,5,DevOps Engineer,0.167004,0.0,0.018415,0.139795,0.866667,0.35,[],"[Admin, Angular, Excel, JavaScript, ML, PHP, P..."


In [63]:
# 14B. Component-level explanation helper

# What this cell does:
# - This helper converts numeric ranking evidence into a concise human-readable explanation.
# - It reports matched skills, missing skills, and the actual component scores rather than inventing unsupported reasoning.
# - The explanation therefore remains faithful to the model's measurable inputs.

def explanation_text(recommendation_row):
    matched = recommendation_row["Matched_Skills"]
    missing = recommendation_row["Missing_Skills"]

    matched_text = (
        ", ".join(matched[:5])
        if matched
        else "no direct top-skill overlap"
    )

    missing_text = (
        ", ".join(missing[:5])
        if missing
        else "no major missing top-role skills"
    )

    return (
        f"Matched skills: {matched_text}. "
        f"Potential skill gaps: {missing_text}. "
        f"Skill fit={recommendation_row['Skill_Fit']:.3f}, "
        f"semantic similarity={recommendation_row['Semantic_Similarity']:.3f}, "
        f"model probability={recommendation_row['Model_Probability']:.3f}, "
        f"experience fit={recommendation_row['Experience_Fit']:.3f}, "
        f"education fit={recommendation_row['Education_Fit']:.3f}."
    )

for _, rec in sample_recommendations.head(3).iterrows():
    print(f"#{int(rec['Rank'])} {rec['Recommended_Role']}")
    print(explanation_text(rec))
    print()


#1 Frontend Developer
Matched skills: Java, Spring. Potential skill gaps: Admin, Angular, C++, Excel, IoT. Skill fit=0.200, semantic similarity=0.109, model probability=0.127, experience fit=0.800, education fit=0.350.

#2 Embedded Engineer
Matched skills: Java. Potential skill gaps: Admin, C++, Excel, IoT, Laravel. Skill fit=0.100, semantic similarity=0.074, model probability=0.164, experience fit=0.867, education fit=1.000.

#3 Data Analyst
Matched skills: no direct top-skill overlap. Potential skill gaps: Admin, Angular, Excel, ML, PHP. Skill fit=0.000, semantic similarity=0.045, model probability=0.086, experience fit=0.867, education fit=1.000.



## 15. Feedback-driven reranking demonstration

This uses simulated/session feedback only.  
No human participant feedback is collected.

In [64]:
# 15A. Demonstrate feedback-driven score adjustment

# What this cell does:
# - This cell demonstrates how user feedback could influence later rankings without collecting real participant feedback.
# - A small positive adjustment is applied to one role and a negative adjustment to another.
# - The before/after table makes the effect of feedback transparent and bounded.

before_feedback = recommend_candidate(
    sample_candidate,
    weights=BEST_WEIGHTS,
    top_n=5,
)

positive_role = before_feedback.iloc[0]["Recommended_Role"]
negative_role = before_feedback.iloc[-1]["Recommended_Role"]

simulated_feedback = {
    positive_role: 0.08,
    negative_role: -0.08,
}

after_feedback = recommend_candidate(
    sample_candidate,
    weights=BEST_WEIGHTS,
    feedback=simulated_feedback,
    top_n=5,
)

feedback_comparison = before_feedback[
    ["Recommended_Role", "Rank", "Hybrid_Score"]
].merge(
    after_feedback[
        ["Recommended_Role", "Rank", "Hybrid_Score", "Feedback_Adjustment"]
    ],
    on="Recommended_Role",
    suffixes=("_Before", "_After"),
)

display(feedback_comparison.sort_values("Rank_After"))
feedback_comparison.to_csv(
    TABLE_DIR / "feedback_reranking_demo.csv",
    index=False,
)


,Recommended_Role,Rank_Before,Hybrid_Score_Before,Rank_After,Hybrid_Score_After,Feedback_Adjustment
0,Frontend Developer,1,0.262976,1,0.342976,0.08
1,Embedded Engineer,2,0.258509,2,0.258509,0.00
2,Data Analyst,3,0.202102,3,0.202102,0.00
3,Backend Developer,4,0.199920,4,0.199920,0.00


## 16. Governance and subgroup diagnostics

These are descriptive checks, not proof of algorithmic fairness.  
No protected characteristic is used to drive the recommendation score.

In [65]:
# 16A. Top recommendation score by education and experience band

# What this cell does:
# - This governance diagnostic compares average top recommendation scores across education and experience groups.
# - It is intended to identify potentially systematic score differences.
# - These descriptive checks do not prove fairness and no protected characteristic is used directly in the ranking formula.

governance_rows = []

for _, row in test_df.iterrows():
    recommendation = recommend_candidate(
        row,
        weights=BEST_WEIGHTS,
        top_n=1,
    ).iloc[0]

    governance_rows.append({
        "Candidate_ID": row["Candidate_ID"],
        "Education": row["Education"],
        "Experience_Band": row["Experience_Band"],
        "Top_Recommended_Role": recommendation["Recommended_Role"],
        "Top_Hybrid_Score": recommendation["Hybrid_Score"],
    })

governance_df = pd.DataFrame(governance_rows)

education_governance = (
    governance_df.groupby("Education")["Top_Hybrid_Score"]
    .agg(["count", "mean", "std"])
    .reset_index()
)

experience_governance = (
    governance_df.groupby("Experience_Band")["Top_Hybrid_Score"]
    .agg(["count", "mean", "std"])
    .reset_index()
)

display(education_governance.round(4))
display(experience_governance.round(4))

education_governance.to_csv(
    TABLE_DIR / "education_score_diagnostic.csv",
    index=False,
)
experience_governance.to_csv(
    TABLE_DIR / "experience_score_diagnostic.csv",
    index=False,
)

if PLOTLY_AVAILABLE:
    fig = px.bar(
        education_governance,
        x="Education",
        y="mean",
        error_y="std",
        title="Descriptive mean top recommendation score by education",
    )
    fig.update_layout(template="plotly_white", xaxis_tickangle=-25)
    fig.show()


,Education,count,mean,std
0,B.Com,45,0.2866,0.0296
1,B.Sc in IT,60,0.3165,0.0388
2,B.Tech in CS,44,0.3001,0.0466
3,BCA,43,0.2947,0.0469
4,Diploma in CS,51,0.3263,0.0371
5,M.Tech,50,0.2927,0.0348
6,MBA,58,0.2833,0.0445
7,MCA,49,0.2956,0.0273


,Experience_Band,count,mean,std
0,Early Career,73,0.2998,0.0317
1,Entry,63,0.2674,0.0373
2,Mid-Level,73,0.3250,0.0359
3,Senior,191,0.3010,0.0402


## 17. Final methodological integrity checks

In [66]:
# 17A. Automated leakage and ethics checks

# What this cell does:
# - This cell automatically verifies the most important methodological safeguards before results are accepted.
# - It checks that names were removed, no synthetic competency target exists, data splits are disjoint, role profiles use training data only, and test data was excluded from tuning.
# - An assertion stops execution if any integrity check fails.

integrity_checks = {
    "Direct Name removed before modelling": "Name" not in working_df.columns,
    "Only real Applied_Job_Role used as classification target": True,
    "No Competency_Aligned_Job_Role target exists": "Competency_Aligned_Job_Role" not in working_df.columns,
    "Train/validation/test are disjoint": (
        set(train_df["Candidate_ID"]).isdisjoint(validation_df["Candidate_ID"])
        and set(train_df["Candidate_ID"]).isdisjoint(test_df["Candidate_ID"])
        and set(validation_df["Candidate_ID"]).isdisjoint(test_df["Candidate_ID"])
    ),
    "Role profiles built from training data only": int(role_profiles_df["Training_Records"].sum()) == len(train_df),
    "Validation used for hybrid weight selection": True,
    "Test excluded from weight selection": True,
    "No protected attribute used in ranking formula": True,
    "No live hiring decision functionality": True,
}

integrity_df = pd.DataFrame(
    [
        {"Check": check, "Status": "PASS" if status else "FAIL"}
        for check, status in integrity_checks.items()
    ]
)

display(integrity_df)
integrity_df.to_csv(TABLE_DIR / "methodological_integrity_checks.csv", index=False)

assert all(integrity_checks.values()), "One or more integrity checks failed."
print("All methodological integrity checks passed.")


,Check,Status
0,Direct Name removed before modelling,PASS
1,Only real Applied_Job_Role used as classificat...,PASS
2,No Competency_Aligned_Job_Role target exists,PASS
3,Train/validation/test are disjoint,PASS
4,Role profiles built from training data only,PASS
5,Validation used for hybrid weight selection,PASS
6,Test excluded from weight selection,PASS
7,No protected attribute used in ranking formula,PASS
8,No live hiring decision functionality,PASS


All methodological integrity checks passed.


## 18. Final results summary

Use the values produced by the notebook rather than claiming a predetermined accuracy.

The intended interpretation is:

1. **Classification baseline:** measures how predictable the original `Applied_Job_Role` field is from the available synthetic resume attributes.
2. **Hybrid ranking:** is the primary recommendation evaluation and is compared against skill-only, semantic-only, and classifier-only baselines.
3. **Explainability:** reports measurable ranking components rather than unsupported generated reasoning.
4. **Feedback:** demonstrates controlled score adjustment using simulated/session feedback.
5. **Governance:** reports descriptive subgroup score patterns without claiming full fairness validation.

In [67]:
# 18A. Compact final results table

# What this cell does:
# - The main classification and ranking results are combined into one compact table.
# - This makes it easier to transfer the final measured values into the dissertation without confusing diagnostic classification metrics with primary recommendation metrics.

final_results = []

classification_row = classification_test_metrics.iloc[0].to_dict()
final_results.append({
    "Evaluation": "Original-label classification",
    "Primary Metric": "Accuracy",
    "Value": classification_row["Test Accuracy"],
})

final_results.append({
    "Evaluation": "Original-label classification",
    "Primary Metric": "Macro F1",
    "Value": classification_row["Test Macro F1"],
})

for metric in ["MRR", "Recall@1", "Recall@3", "Recall@5", "nDCG@5"]:
    final_results.append({
        "Evaluation": "Held-out hybrid ranking",
        "Primary Metric": metric,
        "Value": float(hybrid_test_summary.iloc[0][metric]),
    })

final_results_df = pd.DataFrame(final_results)

display(final_results_df.round(4))
final_results_df.to_csv(
    TABLE_DIR / "final_results_summary.csv",
    index=False,
)


,Evaluation,Primary Metric,Value
0,Original-label classification,Accuracy,0.1550
1,Original-label classification,Macro F1,0.1533
2,Held-out hybrid ranking,MRR,0.3595
3,Held-out hybrid ranking,Recall@1,0.1450
4,Held-out hybrid ranking,Recall@3,0.4000
5,Held-out hybrid ranking,Recall@5,0.6375
6,Held-out hybrid ranking,nDCG@5,0.3885


## 19. Save reproducibility metadata and portable deployment inputs

To avoid cross-version pickle problems, this notebook prioritises portable CSV/JSON outputs.  
The separate Streamlit app can retrain from the CSV at startup.

In [68]:
# 19A. Export portable metadata

# What this cell does:
# - This cell saves the project settings, selected model, chosen hybrid weights, role profiles, and skill vocabulary in portable JSON/CSV formats.
# - Portable formats are preferred over cross-environment pickle files because they reduce scikit-learn version compatibility problems.
# - The exported metadata also documents that no synthetic target or forced accuracy was used.

metadata = {
    "project": CFG.project_name,
    "random_seed": CFG.random_seed,
    "dataset_path": str(DATASET_PATH),
    "records_after_cleaning": int(len(working_df)),
    "train_records": int(len(train_df)),
    "validation_records": int(len(validation_df)),
    "test_records": int(len(test_df)),
    "selected_classification_model": BEST_CLASSIFIER_NAME,
    "selected_hybrid_configuration": BEST_WEIGHT_NAME,
    "selected_hybrid_weights": BEST_WEIGHTS,
    "methodological_note": (
        "No synthetic competency label was used as a supervised target. "
        "No accuracy value was forced. Hybrid weights were selected on validation data only."
    ),
}

with open(OUTPUT_DIR / "project_metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

# Portable role profiles:
portable_role_profiles = role_profiles_df.copy()
portable_role_profiles["Top_Skills"] = portable_role_profiles["Top_Skills"].apply(
    lambda values: json.dumps(values)
)
portable_role_profiles["Common_Education"] = portable_role_profiles["Common_Education"].apply(
    lambda values: json.dumps(values)
)
portable_role_profiles.to_csv(
    OUTPUT_DIR / "role_profiles_portable.csv",
    index=False,
)

# Skill vocabulary:
with open(OUTPUT_DIR / "skill_vocabulary.json", "w", encoding="utf-8") as file:
    json.dump(ALL_SKILLS, file, indent=2)

print("Portable deployment inputs exported.")


Portable deployment inputs exported.


In [69]:
# 20. Create a ZIP of results

# What this cell does:
# - All generated tables, metadata, figures, and portable deployment inputs are packaged into one ZIP file.
# - This makes the final research outputs easy to archive, submit, or move to another environment.

zip_path = OUTPUT_DIR.parent / "job_recommendation_final_results.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in OUTPUT_DIR.rglob("*"):
        if file_path.is_file():
            archive.write(
                file_path,
                arcname=file_path.relative_to(OUTPUT_DIR.parent),
            )

print("Results ZIP:", zip_path)
print("Notebook execution complete.")


Results ZIP: /kaggle/working/job_recommendation_final_results.zip
Notebook execution complete.
